In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/gold/fact_order_shipment"

In [0]:
df_silver_shipment = spark.readStream.format("delta").option("readChangeFeed", "true").table(f"{catalog_name}.silver.slv_order_shipment")


In [0]:
df_gold_shipment = df_silver_shipment.withColumn("carrier_group", F.when(F.col("carrier").isin(["ECOMMERCE","DELHIVERY","XPRESSBEES","BLUEDART"]), "DOMESTIC").otherwise("INTERNATIONAL"))

In [0]:
df_gold_shipment=df_gold_shipment.withColumn("is_weekend_shipment", F.when(F.dayofweek(F.col("order_dt")).isin([1,7]), 1).otherwise(0))

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_shipment"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.shipment_id = batch_table.shipment_id" ,
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    


In [0]:
df_gold_shipment.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()